# 02. Aggregate 1 km grid to 5 km and 10 km

Block-mean aggregation of the 1 km feature parquets from
`01_build_grid.ipynb` to 5 km (5x5 blocks) and 10 km (10x10 blocks).
Output schema and column order match the 1 km input.


## 1. Setup


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
from joblib import Parallel, delayed

ROOT = Path('../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

s3 = boto3.client('s3')
N_CORES = os.cpu_count() or 1
print(f'cwd       : {os.getcwd()}')
print(f'CPU cores : {N_CORES}')


## 2. Config


In [ ]:
SOURCE_RES_M   = 1000
TARGET_RES_M   = [5000, 10000]

S3_BUCKET      = 'thesis-data-ismaktam'
SRC_PREFIX     = 'dataset/grid_1km'
LOCAL_SRC_DIR  = Path('results/dataset/grid_1km')
LOCAL_OUT_DIR  = Path('results/dataset')
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Columns that live in the static parquet (grid_1km_static.parquet)
# and are averaged when building the coarse grid in section 3.
STATIC_AVG_COLS = [
    'latitude', 'longitude', 'x_proj', 'y_proj', 'elevation_m',
    'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa',
]
# Per-day dynamic columns averaged inside each coarse cell in section 4.
DYNAMIC_AVG_COLS = ['idw', 'gos', 'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09']

SKIP_EXISTING = True

print(f'source : {SOURCE_RES_M} m')
print(f'targets: {TARGET_RES_M} m')


## 3. Build per-cell aggregation index


In [ ]:
STATIC_PATH = Path('results/dataset/grid_1km_static.parquet')
if not STATIC_PATH.exists():
    s3.download_file(S3_BUCKET, 'dataset/grid_1km_static.parquet', str(STATIC_PATH))
static = pd.read_parquet(STATIC_PATH)
print(f'1 km static cells: {len(static):,}')

x_min, x_max = static['x_proj'].min(), static['x_proj'].max()
y_min, y_max = static['y_proj'].min(), static['y_proj'].max()

agg_index = {}
agg_static = {}
for tgt in TARGET_RES_M:
    bx = ((static['x_proj'] - x_min) // tgt).astype(np.int64)
    by = ((static['y_proj'] - y_min) // tgt).astype(np.int64)
    block = list(zip(bx, by))
    static_tmp = static.assign(_bx=bx, _by=by)

    coarse = (static_tmp
              .groupby(['_bx', '_by'], as_index=False)[STATIC_AVG_COLS]
              .mean())
    coarse['cell_id'] = np.arange(len(coarse), dtype=np.int64)

    blk_to_cell = coarse.set_index(['_bx', '_by'])['cell_id'].to_dict()
    static_tmp['coarse_cell_id'] = [blk_to_cell[(a, b)] for a, b in block]
    agg_index[tgt] = static_tmp[['cell_id', 'coarse_cell_id']]
    agg_static[tgt] = coarse.drop(columns=['_bx', '_by'])

    coarse_path = LOCAL_OUT_DIR / f'grid_{tgt//1000}km_static.parquet'
    agg_static[tgt].to_parquet(coarse_path, index=False)
    s3.upload_file(str(coarse_path), S3_BUCKET,
                   f'dataset/grid_{tgt//1000}km_static.parquet')
    print(f'  {tgt} m: {len(coarse):,} coarse cells  '
          f'-> s3://{S3_BUCKET}/dataset/grid_{tgt//1000}km_static.parquet')


## 4. Aggregate monthly parquets


In [ ]:
def _s3_key_exists(key):
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except s3.exceptions.ClientError:
        return False

month_keys = []
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=f'{SRC_PREFIX}/'):
    for obj in page.get('Contents', []):
        if obj['Key'].endswith('.parquet'):
            month_keys.append(obj['Key'])
month_keys.sort()
print(f'months in source: {len(month_keys)}')

DYNAMIC_AVG_COLS = ['idw', 'gos', 'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09']
COL_ORDER = [
    'datetime', 'latitude', 'longitude', 'x_proj', 'y_proj', 'elevation_m',
    'idw', 'gos', 'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
    'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa',
    'rainfall', 'rainfall_int', 'station_id', 'cell_id', 'fold',
]

def _process_month(src_key, tgt, tgt_prefix, idx, coarse_static, out_local_dir):
    import boto3 as _boto3
    _s3 = _boto3.client('s3')
    month_name = Path(src_key).name
    out_key = f'{tgt_prefix}/{month_name}'
    if SKIP_EXISTING:
        try:
            _s3.head_object(Bucket=S3_BUCKET, Key=out_key)
            return f'skip {tgt//1000}km/{month_name}'
        except _s3.exceptions.ClientError:
            pass

    out_local = out_local_dir / month_name
    local_src = LOCAL_OUT_DIR / 'grid_1km' / month_name
    local_src.parent.mkdir(parents=True, exist_ok=True)
    _s3.download_file(S3_BUCKET, src_key, str(local_src))

    df = pd.read_parquet(local_src).merge(idx, on='cell_id', how='left')
    grp = (df.groupby(['datetime', 'coarse_cell_id'], as_index=False)
             [DYNAMIC_AVG_COLS].mean())
    grp = grp.rename(columns={'coarse_cell_id': 'cell_id'})
    grp = grp.merge(coarse_static, on='cell_id', how='left')
    grp['rainfall']     = np.float64('nan')
    grp['rainfall_int'] = np.int32(-1)
    grp['station_id']   = grp['cell_id'].astype(str)
    grp['fold']         = np.int8(-2)
    for c in DYNAMIC_AVG_COLS:
        grp[c] = grp[c].astype(np.float32)
    grp = grp[COL_ORDER]
    grp.to_parquet(out_local, index=False)
    _s3.upload_file(str(out_local), S3_BUCKET, out_key)
    return f'{tgt//1000}km / {month_name}: {len(grp):>8,} rows'


for tgt in TARGET_RES_M:
    out_local_dir = LOCAL_OUT_DIR / f'grid_{tgt//1000}km'
    out_local_dir.mkdir(parents=True, exist_ok=True)
    tgt_prefix = f'dataset/grid_{tgt//1000}km'

    idx = agg_index[tgt][['cell_id', 'coarse_cell_id']]
    coarse_static = agg_static[tgt]

    msgs = Parallel(n_jobs=N_CORES, backend='loky', verbose=5,
                    pre_dispatch='2*n_jobs')(
        delayed(_process_month)(src_key, tgt, tgt_prefix, idx,
                                coarse_static, out_local_dir)
        for src_key in month_keys
    )
    for m in msgs:
        print(' ', m)
